# Case Study 1 – Analysis
Answers the business questions using the clean tables, mostly with SQL.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd, sqlite3, shutil
BASE = '/content/drive/MyDrive/cs1-device-safety'
C = f'{BASE}/data/clean'

shutil.copy(f'{C}/maude_rehab.db', '/content/maude_rehab.db')
con = sqlite3.connect('/content/maude_rehab.db')

def sql(query):
    return pd.read_sql(query, con)

print(sql("SELECT name FROM sqlite_master WHERE type = 'table'"))
print(sql("SELECT COUNT(*) AS reports FROM reports"))

Mounted at /content/drive
               name
0           reports
1   device_problems
2  patient_problems
3        narratives
   reports
0      477


The business question: "If I sell this device, what injuries should I warn about and train for?"

In [2]:
sql("""
WITH counts AS (
    SELECT device_group,
           patient_problem,
           COUNT(DISTINCT MDR_REPORT_KEY) AS reports
    FROM patient_problems
    GROUP BY device_group, patient_problem
),
ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY device_group ORDER BY reports DESC) AS rank
    FROM counts
)
SELECT device_group, rank, patient_problem, reports
FROM ranked
WHERE rank <= 5
ORDER BY device_group, rank
""")

,device_group,rank,patient_problem,reports
0,Powered exercise equipment (incl. CPM),1,"No Clinical Signs, Symptoms or Conditions",13
1,Powered exercise equipment (incl. CPM),2,Injury,4
2,Powered exercise equipment (incl. CPM),3,Insufficient Information,4
3,Powered exercise equipment (incl. CPM),4,Bone Fracture(s),3
4,Powered exercise equipment (incl. CPM),5,No Known Impact Or Consequence To Patient,3
5,Powered exoskeleton,1,Bone Fracture(s),33
6,Powered exoskeleton,2,Swelling,5
7,Powered exoskeleton,3,Bruise/Contusion,3
8,Powered exoskeleton,4,Limb Fracture,3
9,Powered exoskeleton,5,"No Clinical Signs, Symptoms or Conditions",3


Analysis 1b – Top real harms per device (non-harm entries removed), as % of the device's reports.

In [3]:
NOT_HARM = ['No Clinical Signs%', 'No Known Impact%', 'No Consequences%',
            'No Patient Involvement%', 'No Information%', 'Insufficient Information%']
REAL_HARM = ' AND '.join(f"patient_problem NOT LIKE '{p}'" for p in NOT_HARM)

top_harms = sql(f"""
WITH counts AS (
    SELECT device_group, patient_problem,
           COUNT(DISTINCT MDR_REPORT_KEY) AS n
    FROM patient_problems
    WHERE {REAL_HARM}
    GROUP BY device_group, patient_problem
),
ranked AS (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY device_group ORDER BY n DESC) AS rank
    FROM counts
),
totals AS (
    SELECT device_group, COUNT(*) AS total FROM reports GROUP BY device_group
)
SELECT r.device_group, r.rank, r.patient_problem, r.n AS reports,
       ROUND(100.0 * r.n / t.total, 1) AS pct_of_device_reports
FROM ranked r JOIN totals t ON r.device_group = t.device_group
WHERE r.rank <= 5
ORDER BY r.device_group, r.rank
""")
print(top_harms.to_string(index=False), '\n')

print(sql(f"""
SELECT device_group,
       COUNT(DISTINCT MDR_REPORT_KEY) AS reports,
       COUNT(DISTINCT CASE WHEN {REAL_HARM} THEN MDR_REPORT_KEY END) AS reports_with_real_harm
FROM patient_problems
GROUP BY device_group
""").to_string(index=False))

                          device_group  rank                                                             patient_problem  reports  pct_of_device_reports
Powered exercise equipment (incl. CPM)     1                                                                      Injury        4                   10.0
Powered exercise equipment (incl. CPM)     2                                                            Bone Fracture(s)        3                    7.5
Powered exercise equipment (incl. CPM)     3                                                                        Pain        3                    7.5
Powered exercise equipment (incl. CPM)     4                                         Ambulation or Postural Difficulties        2                    5.0
Powered exercise equipment (incl. CPM)     5                                              Hemorrhage/Blood Loss/Bleeding        2                    5.0
                   Powered exoskeleton     1                                      

Analysis 1c – List all real harm terms, to group them into clinical categories.

In [4]:
print(sql(f"""
SELECT patient_problem, COUNT(DISTINCT MDR_REPORT_KEY) AS reports
FROM patient_problems
WHERE {REAL_HARM}
GROUP BY patient_problem
ORDER BY reports DESC
""").to_string(index=False))

                                                            patient_problem  reports
                                                                    Burn(s)       59
                                     Partial thickness (Second Degree) Burn       54
                                                                       Pain       44
                                                           Bone Fracture(s)       36
                                         Full thickness (Third Degree) Burn       24
                                                             Electric Shock       24
                                                              Burn, Thermal       23
                                                                   Erythema       17
                                            Superficial (First Degree) Burn       16
                                                                    Blister       12
                                                                 

Analysis 2 – Harm categories: % of each device's reports mentioning each category.

In [5]:
reports = pd.read_parquet(f'{C}/reports.parquet')
pp = pd.read_parquet(f'{C}/patient_problems.parquet')
dp = pd.read_parquet(f'{C}/device_problems.parquet')
narr = pd.read_parquet(f'{C}/narratives.parquet')
totals = reports.groupby('device_group').size()

CATEGORIES = {
    'Burn / skin injury': ['Burn(s)', 'Partial thickness (Second Degree) Burn', 'Full thickness (Third Degree) Burn',
        'Burn, Thermal', 'Superficial (First Degree) Burn', 'Lead(s), Burn(s) From', 'Caustic/Chemical Burns',
        'Blister', 'Erythema', 'Rash', 'Skin Irritation', 'Skin Inflammation', 'Skin Inflammation/ Irritation',
        'Itching Sensation', 'Skin Discoloration', 'Skin Erosion', 'Peeling', 'Scar Tissue', 'Scarring',
        'Abnormal scarring', 'Local Reaction', 'Hypersensitivity/Allergic reaction', 'Burning Sensation',
        'Skin Burning Sensation', 'Tissue Damage'],
    'Electric shock / unwanted stimulation': ['Electric Shock', 'Shock', 'Shock from Patient Lead(s)',
        'Undesired Nerve Stimulation', 'Device Overstimulation of Tissue', 'Muscle Stimulation', 'Twitching',
        'Tingling', 'Cramp(s) /Muscle Spasm(s)', 'Muscle Spasm(s)'],
    'Pain / discomfort': ['Pain', 'Discomfort', 'Neck Pain', 'Back pain', 'Headache', 'Arthralgia', 'Myalgia',
        'Eye Pain', 'Ear Pain', 'Chest Pain'],
    'Fall / fracture / musculoskeletal': ['Bone Fracture(s)', 'Limb Fracture', 'Hip Fracture', 'Fracture, Arm',
        'Fall', 'Bruise/Contusion', 'Laceration(s)', 'Head Injury', 'Spinal Cord Injury', 'Muscle/Tendon Damage',
        'Loss of Range of Motion', 'Ambulation or Postural Difficulties', 'Abrasion', 'Swelling',
        'Swelling/ Edema', 'Joint Swelling', 'Hemorrhage/Blood Loss/Bleeding'],
    'Serious systemic (cardiac, neuro, other)': ['Atrial Fibrillation', 'Atrial Flutter', 'Arrhythmia',
        'Tachycardia', 'Palpitations', 'Cardiac Arrest', 'Heart Failure', 'Seizures, Grand-Mal',
        'Convulsion, Tonic', 'Transient Ischemic Attack', 'Thrombosis', 'Thrombosis/Thrombus', 'Sepsis',
        'Respiratory Acidosis', 'Loss of consciousness', 'Abortion'],
}
NOT_HARM = ('No Clinical Signs', 'No Known Impact', 'No Consequences', 'No Patient Involvement',
            'No Information', 'Insufficient Information', 'Appropriate Clinical Signs', 'No Code Available',
            'Patient Problem/Medical Problem', 'Inadequate Pain Relief')

term_to_cat = {t: c for c, terms in CATEGORIES.items() for t in terms}
pp['patient_problem'] = pp['patient_problem'].fillna('')
harm_rows = pp[(pp['patient_problem'] != '') & ~pp['patient_problem'].str.startswith(NOT_HARM)].copy()
harm_rows['harm_category'] = harm_rows['patient_problem'].map(term_to_cat).fillna('Other')
cat = harm_rows.drop_duplicates(['MDR_REPORT_KEY', 'harm_category'])

harm_counts = cat.groupby(['device_group', 'harm_category']).size().unstack(fill_value=0)
harm_pct = (harm_counts.div(totals, axis=0) * 100).round(1)
print(harm_pct.T.to_string())

device_group                              Powered exercise equipment (incl. CPM)  Powered exoskeleton  Powered muscle stimulator  TENS (over-the-counter)  TENS (prescription)
harm_category                                                                                                                                                                 
Burn / skin injury                                                           2.5                  0.0                       34.2                     48.5                 62.0
Electric shock / unwanted stimulation                                        0.0                  3.7                       16.4                     15.5                 14.6
Fall / fracture / musculoskeletal                                           27.5                 81.5                        6.8                      5.2                  4.2
Other                                                                       17.5                  5.6                       2

Analysis 3 – Severity, home use, corrective action per device; and severe harm with vs without home/lay use.

In [6]:
g = reports.groupby('device_group')
severity = pd.DataFrame({
    'reports': g.size(),
    'serious_%': (g['serious_outcome'].mean() * 100).round(1),
    'severe_%': (g['severe_outcome'].mean() * 100).round(1),
    'home_or_lay_use_%': (g['home_or_lay_use'].mean() * 100).round(1),
    'corrective_action_%': (g['corrective_action'].mean() * 100).round(1),
    'recalls': g['recall'].sum()})
print(severity.to_string(), '\n')

home_vs = (reports.groupby(['device_group', 'home_or_lay_use'])['severe_outcome']
           .agg(reports='size', severe_pct='mean'))
home_vs['severe_pct'] = (home_vs['severe_pct'] * 100).round(1)
print(home_vs.to_string())

                                        reports  serious_%  severe_%  home_or_lay_use_%  corrective_action_%  recalls
device_group                                                                                                         
Powered exercise equipment (incl. CPM)       40       37.5      27.5               12.5                 20.0        2
Powered exoskeleton                          54       77.8      13.0               46.3                 42.6        8
Powered muscle stimulator                    73       26.0      15.1               27.4                 16.4        0
TENS (over-the-counter)                      97       30.9      18.6               76.3                  9.3        0
TENS (prescription)                         213       25.4       4.7               25.4                  2.8        0 

                                                        reports  severe_pct
device_group                           home_or_lay_use                     
Powered exercise equ

Analysis 4 – Reports per 2-year period per device (2024–26 includes a partial 2026).

In [7]:
reports['period'] = pd.cut(reports['year'], bins=[2015, 2017, 2019, 2021, 2023, 2026],
                           labels=['2016-17', '2018-19', '2020-21', '2022-23', '2024-26'])
trend = reports.pivot_table(index='period', columns='device_group', values='MDR_REPORT_KEY',
                            aggfunc='count', fill_value=0, observed=False)
print(trend.to_string())

device_group  Powered exercise equipment (incl. CPM)  Powered exoskeleton  Powered muscle stimulator  TENS (over-the-counter)  TENS (prescription)
period                                                                                                                                            
2016-17                                           10                   12                         10                       34                   90
2018-19                                            4                   21                         13                       23                   28
2020-21                                            2                    5                         16                       15                   15
2022-23                                           15                    3                         12                        6                   43
2024-26                                            9                   13                         22                  

Analysis 5 – Supplier profiles: Dongdixin vs other prescription TENS; ReWalk vs other exoskeletons.

In [8]:
DEEP = ['Partial thickness (Second Degree) Burn', 'Full thickness (Third Degree) Burn']

def profile(df):
    keys = set(df['MDR_REPORT_KEY'])
    c = cat[cat['MDR_REPORT_KEY'].isin(keys)]
    n = len(df)
    pct = lambda s: round(100 * s / n, 1)
    return pd.Series({
        'reports': n,
        'burn_or_skin_%': pct(c.loc[c.harm_category == 'Burn / skin injury', 'MDR_REPORT_KEY'].nunique()),
        'deep_burn_%': pct(harm_rows.loc[harm_rows['MDR_REPORT_KEY'].isin(keys) &
                                         harm_rows['patient_problem'].isin(DEEP), 'MDR_REPORT_KEY'].nunique()),
        'fracture_or_fall_%': pct(c.loc[c.harm_category == 'Fall / fracture / musculoskeletal', 'MDR_REPORT_KEY'].nunique()),
        'severe_%': pct(df['severe_outcome'].sum()),
        'home_or_lay_%': pct(df['home_or_lay_use'].sum()),
        'filed_by_manufacturer_%': pct((df['report_source'] == 'Manufacturer').sum())})

rx = reports[reports.device_group == 'TENS (prescription)']
exo = reports[reports.device_group == 'Powered exoskeleton']
DDX, RW = 'SHENZHEN DONGDIXIN TECHNOLOGY', 'REWALK ROBOTICS'
supplier = pd.DataFrame({
    'Dongdixin (Rx TENS)': profile(rx[rx.manufacturer == DDX]),
    'Other Rx TENS': profile(rx[rx.manufacturer != DDX]),
    'ReWalk (exoskeleton)': profile(exo[exo.manufacturer == RW]),
    'Other exoskeletons': profile(exo[exo.manufacturer != RW])}).T
print(supplier.to_string(), '\n')
print('Dongdixin reports by year:\n', rx[rx.manufacturer == DDX].groupby('year').size().to_string())

                      reports  burn_or_skin_%  deep_burn_%  fracture_or_fall_%  severe_%  home_or_lay_%  filed_by_manufacturer_%
Dongdixin (Rx TENS)     123.0            79.7         45.5                 5.7       3.3           27.6                     29.3
Other Rx TENS            90.0            37.8          4.4                 2.2       6.7           22.2                     18.9
ReWalk (exoskeleton)     37.0             0.0          0.0                86.5       8.1           64.9                     97.3
Other exoskeletons       17.0             0.0          0.0                70.6      23.5            5.9                     76.5 

Dongdixin reports by year:
 year
2016    29
2017    51
2018    16
2019     2
2021     1
2022     1
2023     8
2024     4
2025     5
2026     6


Analysis 6 – Top device problems per device (non-informative codes removed).

In [9]:
NOT_INFO = ('Adverse Event Without Identified Device or Use Problem', 'Insufficient Information',
            'No Apparent Adverse Event', 'Appropriate Term/Code Not Available', 'Unknown')
dp['device_problem'] = dp['device_problem'].fillna('Unlisted problem code')

no_device_fault = (dp[dp['device_problem'].str.startswith('Adverse Event Without Identified')]
                   .groupby('device_group')['MDR_REPORT_KEY'].nunique())
print('% of reports: harm with NO device fault identified\n',
      (100 * no_device_fault / totals).round(1).to_string(), '\n')

dpi = dp[~dp['device_problem'].str.startswith(NOT_INFO)]
top_dp = dpi.groupby(['device_group', 'device_problem'])['MDR_REPORT_KEY'].nunique().rename('reports').reset_index()
top_dp['pct_of_device_reports'] = (100 * top_dp['reports'] / top_dp['device_group'].map(totals)).round(1)
top_dp = top_dp.sort_values(['device_group', 'reports'], ascending=[True, False]).groupby('device_group').head(5)
print(top_dp.to_string(index=False))

% of reports: harm with NO device fault identified
 device_group
Powered exercise equipment (incl. CPM)     7.5
Powered exoskeleton                       33.3
Powered muscle stimulator                 16.4
TENS (over-the-counter)                   27.8
TENS (prescription)                       41.3 

                          device_group                             device_problem  reports  pct_of_device_reports
Powered exercise equipment (incl. CPM)                                      Break        4                   10.0
Powered exercise equipment (incl. CPM)                      Use of Device Problem        4                   10.0
Powered exercise equipment (incl. CPM)  Improper or Incorrect Procedure or Method        3                    7.5
Powered exercise equipment (incl. CPM)               Physical Resistance/Sticking        3                    7.5
Powered exercise equipment (incl. CPM)                        Failure to Power Up        2                    5.0
              

Analysis 7 – Narratives: % of event descriptions mentioning key terms, plus sample stories to read.

In [11]:
raw = narr[narr['TEXT_TYPE_CODE'] == 'D'].groupby('MDR_REPORT_KEY')['FOI_TEXT'].apply(lambda x: ' '.join(x.dropna()))
text = raw.str.lower()
KEYWORDS = {'burn': r'burn', 'blister': r'blister', 'electrode / pad': r'electrode|\bpads?\b',
            'sleep / asleep': r'asleep|\bsleep', 'overnight / hours': r'overnight|all night|\bhours\b',
            'fell / fall': r'\bfell\b|\bfall', 'fracture': r'fractur', 'strap': r'strap',
            'overheat / hot / melt': r'overheat|\bhot\b|melt', 'instructions / manual': r'instruction|manual'}
kw = pd.DataFrame({k: text.str.contains(p, regex=True) for k, p in KEYWORDS.items()})
kw['device_group'] = reports.set_index('MDR_REPORT_KEY')['device_group'].reindex(kw.index)
print((kw.groupby('device_group').mean() * 100).round(1).T.to_string(), '\n')

def show(keys, title, n=4, width=500):
    print(f'===== {title}')
    for k in list(keys)[:n]:
        print('-', raw.get(k, '(no narrative)')[:width], '\n')

show(reports.query("device_group == 'TENS (over-the-counter)' and severe_outcome")['MDR_REPORT_KEY'], 'Severe OTC TENS')
show(harm_rows.query("device_group == 'Powered exoskeleton' and harm_category == 'Fall / fracture / musculoskeletal'")['MDR_REPORT_KEY'].unique(), 'Exoskeleton fractures')
show(harm_rows.loc[harm_rows['patient_problem'].isin(DEEP) &
                   harm_rows['MDR_REPORT_KEY'].isin(rx.loc[rx.manufacturer == DDX, 'MDR_REPORT_KEY']), 'MDR_REPORT_KEY'].unique(),
     'Dongdixin deep burns')

device_group           Powered exercise equipment (incl. CPM)  Powered exoskeleton  Powered muscle stimulator  TENS (over-the-counter)  TENS (prescription)
burn                                                      2.6                  1.9                       35.6                     41.2                 62.0
blister                                                   0.0                  0.0                        4.1                     15.5                 19.2
electrode / pad                                           5.1                  1.9                       24.7                     53.6                 55.4
sleep / asleep                                            0.0                  1.9                        0.0                      7.2                  2.3
overnight / hours                                         2.6                  7.5                        1.4                      3.1                  8.9
fell / fall                                              10.3   

Save result tables for the dashboard.

In [12]:
import os
OUT = f'{BASE}/outputs/tables'
os.makedirs(OUT, exist_ok=True)
severity.to_csv(f'{OUT}/severity.csv')
harm_pct.to_csv(f'{OUT}/harm_categories_pct.csv')
trend.to_csv(f'{OUT}/trend.csv')
supplier.to_csv(f'{OUT}/supplier_profiles.csv')
top_dp.to_csv(f'{OUT}/top_device_problems.csv', index=False)
(kw.groupby('device_group').mean() * 100).round(1).to_csv(f'{OUT}/narrative_keywords_pct.csv')
print('Saved:', os.listdir(OUT))

Saved: ['severity.csv', 'harm_categories_pct.csv', 'trend.csv', 'supplier_profiles.csv', 'top_device_problems.csv', 'narrative_keywords_pct.csv']


Print all results for review.

In [13]:
for f in sorted(os.listdir(OUT)):
    print(f'\n===== {f}')
    print(pd.read_csv(f'{OUT}/{f}').to_string(index=False))


===== harm_categories_pct.csv
                          device_group  Burn / skin injury  Electric shock / unwanted stimulation  Fall / fracture / musculoskeletal  Other  Pain / discomfort  Serious systemic (cardiac, neuro, other)
Powered exercise equipment (incl. CPM)                 2.5                                    0.0                               27.5   17.5               10.0                                       2.5
                   Powered exoskeleton                 0.0                                    3.7                               81.5    5.6                5.6                                       3.7
             Powered muscle stimulator                34.2                                   16.4                                6.8   21.9               20.5                                       5.5
               TENS (over-the-counter)                48.5                                   15.5                                5.2   11.3               12.4       